In [9]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("How does dolphins communicate?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "How does dolphins communicate?" This is a straightforward scientific/biological question about dolphin communication methods.\n\n2.  **Identify Key Concepts**: \n   - Dolphins are highly intelligent marine mammals\n   - They use multiple communication channels\n   - Primary methods: vocalizations (clicks, whistles, burst pulses), body language, tactile communication, and possibly chemical/electroreception (though less prominent)\n   - Functions: social bonding, hunting/navigation, identifying individuals, coordinating group behavior\n   - Key research areas: signature whistles, echolocation, dialects, cultural transmission\n\n3.  **Structure the Response**:\n   - Introduction: Brief overview of dolphin communication complexity\n   - Vocal Communication (main focus)\n     - Clicks (echolocation & social)\n     - Whistles (signature whistles, social bonds)\n     - Burst-pulse sou

In [12]:
from langchain.tools import tool 

@tool 
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tool = model.bind_tools([get_weather])

In [13]:
response = model_with_tool.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    #view the tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in Boston.\n2.  Identify the required tool: `get_weather`\n3.  Extract the parameter: `location` = "Boston"\n4.  Call the tool: `get_weather(location="Boston")`\n5.  Formulate the response based on the tool\'s output. (I will simulate the tool call now)\nWait, I am the AI, I need to output the tool call. I will generate the JSON for the tool call.\nParameters: `{"location": "Boston"}`\nCheck schema: `get_weather` requires `location` (string). Matches.\nProceed. \nOutput matches the required format.✅\nLet\'s generate the tool call. \n`print(json.dumps({"name": "get_weather", "arguments": {"location": "Boston"}}))` -> Actually, the system handles the formatting. I just output the tool call block.\nWait, the prompt says: "If you choose to call a function ONLY reply in the following format with NO suffix: <tool_call>\n<function=example_function_name>\n<

### Tool Execution Loop

In [16]:
#Step1: Model generates the tool calls
messages = [{"role":"user", "content":"What's the weather in Kathmandu?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)


#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)

It's sunny in Kathmandu.


In [17]:
messages

[{'role': 'user', 'content': "What's the weather in Kathmandu?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: Weather in Kathmandu.\n2.  Identify the available tool: `get_weather` with parameter `location`.\n3.  Extract the location: "Kathmandu".\n4.  Call the tool: `get_weather(location="Kathmandu")`.\n5.  Formulate response based on tool output. (I will simulate the tool call now)\nWait, I need to output the tool call first.\nParameters: `{"location": "Kathmandu"}`.\nProceed. \nNo extra fluff. Just the tool call.✅\n', 'tool_calls': [{'id': 'tv7bac8p7', 'function': {'arguments': '{"location":"Kathmandu"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 277, 'total_tokens': 442, 'completion_time': 0.324291214, 'completion_tokens_details': {'reasoning_tokens': 134}, 'prompt_time': 0.024199315, 'prompt_tokens_details': None, 'queue_time': 